# Etapa preliminar — integrador sobre malla discretizada

> ⚠️ **Versión preliminar, descartada.** Este cuaderno corresponde al informe
> [`preliminar-fis205.pdf`](../informe/preliminar-fis205.pdf). El modelo vigente es
> [`modelo-actual.ipynb`](./modelo-actual.ipynb), que resuelve en campos analíticos y **no
> necesita compilar nada**.

Las secciones 3 y 5 de este cuaderno dependen del módulo de extensión `motor_mpd_cpp`, escrito
en C++ y enlazado a Python con pybind11. El resto corre en Python puro.

## Cómo compilar el módulo

El código fuente está en [`preliminar-cpp/`](../preliminar-cpp). Debe permanecer junto a su
`CMakeLists.txt`. Requiere `pybind11` y un compilador de C++17:

```bash
pip install pybind11
```

```bash
cd preliminar-cpp
```

```bash
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
```

```bash
cmake --build build --config Release
```

El módulo resultante queda en `preliminar-cpp/build/` con Makefiles o MinGW, y en
`preliminar-cpp/build/Release/` con el generador de Visual Studio. Las celdas de abajo buscan
en ambas ubicaciones.

**Nota para Windows con MinGW:** Python 3.8 y posteriores no resuelven las DLL del runtime a
través del `PATH`. Si la importación falla con `ImportError: DLL load failed`, hay que declarar
el directorio del compilador antes de importar:

```python
os.add_dll_directory(r"C:\msys64\ucrt64\bin")
```


# 1 · Parámetros maestros

In [ ]:
import numpy as np


#  PARÁMETROS MAESTROS  (única fuente — el resto de celdas lee de aquí)


# --- Constantes físicas (SI) ---
e      = 1.602e-19    # C        - Carga elemental
m_ion  = 6.63e-26     # kg       - Masa del ion Ar+
m_e    = 9.109e-31    # kg       - Masa del electrón
g0     = 9.80665      # m/s^2    - Gravedad estándar (Isp)

# --- Propelente: Argón ---
M_molar = 40.0        # g/mol    - Masa molar
V_ion   = 15.76       # eV       - Potencial de IONIZACIÓN (Ar -> Ar+)   [distinto de la vaina]
T_e     = 1.5         # eV       - Temperatura electrónica
T_i     = 1.5         # eV       - Temperatura iónica (inyección)

# --- Punto de operación ---
I_lab    = 222.0      # A        - Corriente nominal de diseño
B0       = 0.5        # T        - Campo magnético aplicado (garganta)
A_nozzle = 0.04       # m        - Longitud característica de la tobera

# --- Vaina del cátodo (Bohm flotante; NO confundir con V_ion) ---
V_vaina      = (T_e/2)*np.log(m_ion/(2*np.pi*m_e))  # V  ~ 7.0  (caída de vaina)
delta_vaina  = 1.0e-3 # m        - Espesor de la vaina

# --- Dominio de simulación (SI) ---
r_a_max = 5.0e-2      # m        - Radio máximo del ánodo (tope de la campana)
L_sim   = 6.5e-2      # m        - Longitud del dominio axial

# --- Entradas de diseño "Lowcost" (cm, unidades de ingeniería) ---
J_max        = 40.0   # A/cm^2   - Límite de densidad de corriente del W
lambda_ion   = 3.0    # cm       - Camino libre medio de ionización
alpha_loss   = 0.01   # 1/cm     - Coef. de pérdida térmica/viscosa
r_lim        = 5.5    # cm       - Frontera radial de confinamiento
esbeltez_max = 10.0   # -        - Relación L_c/r_c máxima (estructural)
RATIO_LC_LA  = 0.70   # -        - Largo del cátodo respecto al ánodo

# --- Numérico ---
dt         = 5e-10    # s        - Paso temporal (Boris, dt << 1/w_ci)
N_iones    = 4000     # -        - Iones trazados
CADA_FRAME = 500      # pasos    - Submuestreo de la animación

# --- LEGADO: solver (test de malla) — SOLO para el contraste del paper ---


Parametro_Hall = 1.5              
CM2_A_M2       = 1.0e4  

print(f"V_vaina (Bohm) = {V_vaina:.2f} V   |   V_ion = {V_ion:.2f} eV")

# 2 · Geometría óptima del propulsor

In [ ]:

# ============================================================
#  2. OPTIMIZACIÓN GEOMÉTRICA 
# ============================================================
rc_vals = np.linspace(0.1, 4.0, 50)
ra_vals = np.linspace(1.0, 10.0, 50)
La_vals = np.linspace(1.0, 20.0, 50)

mejor = {"eta": -np.inf}
for rc in rc_vals:
    for ra in ra_vals:
        if ra <= rc:
            continue
        for La in La_vals:
            # L_c: razón fija como nominal, límite térmico como piso
            Lc_nominal = RATIO_LC_LA * La
            Lc_termico = I_lab / (2.0 * np.pi * rc * J_max)   # mínimo para J <= J_max
            Lc = max(Lc_nominal, Lc_termico)

            # Restricción estructural (la térmica ya queda garantizada por el max())
            if Lc / rc > esbeltez_max:
                continue

            empuje        = np.log(ra / rc)
            confinamiento = np.exp(-ra / r_lim)
            ionizacion    = 1.0 - np.exp(-La / lambda_ion)
            friccion      = 1.0 - alpha_loss * La
            eta = empuje * confinamiento * ionizacion * friccion

            if eta > mejor["eta"]:
                mejor = {"eta": eta, "rc": rc, "ra": ra, "La": La, "Lc": Lc}

rc, ra, La, Lc = mejor["rc"], mejor["ra"], mejor["La"], mejor["Lc"]   # cm

# --- Geometría canónica para todo lo de abajo ---

r_c, L_c = rc * 1e-2, Lc * 1e-2     # m
r_a, L_a = ra * 1e-2, La * 1e-2     # m

# ============================================================
#  3. PUNTO DE OPERACIÓN  (derivado de geometría + propelente)
# ============================================================
u_ci = np.sqrt(2.0 * e * V_ion / m_ion)            # velocidad crítica de ionización (m/s)
b    = (1e-7) * np.log(ra / rc)                     # mu0/4pi * ln(ra/rc)  (H/m)

mdot_kgs         = b * I_lab**2 / u_ci             # flujo nominal: xi = 1 (u_ex = u_ci)
flujo_masico_mgs = mdot_kgs * 1e6

u_ex = b * I_lab**2 / (flujo_masico_mgs * 1e-6)
xi   = u_ex / u_ci                                 # 1 = nominal, ~2 = onset
Isp  = u_ex / g0                                   # s (componente self-field)
mdot_onset_mgs = (I_lab/1000.0)**2 * np.sqrt(M_molar) / 560.0 * 1000.0
margen_onset   = flujo_masico_mgs / mdot_onset_mgs

# ============================================================
#  4. RESULTADOS
# ============================================================
print("=" * 52)
print(f"   DISEÑO ÓPTIMO  (I = {I_lab:.0f} A, Argón)")
print("=" * 52)
print(f"-> Radio del Cátodo (r_c) : {rc:.3f} cm")
print(f"-> Radio del Ánodo  (r_a) : {ra:.3f} cm")
print(f"-> Longitud Ánodo   (L_a) : {La:.3f} cm")
print(f"-> Longitud Cátodo  (L_c) : {Lc:.3f} cm")
print(f"-> Mérito (eta)           : {mejor['eta']:.4f}")
print("-" * 52)
print(f"-> Flujo Másico (Argón)   : {flujo_masico_mgs:.2f} mg/s")
print(f"-> Punto de operación  xi : {xi:.2f}   (1=nominal, ~2=onset)")
print(f"-> Isp (self-field)       : {Isp:.0f} s")
print(f"-> Margen al onset        : {margen_onset:.1f}x  (umbral {mdot_onset_mgs:.2f} mg/s)")
print("-" * 52)
print(f"-> SI para la sim: r_c={r_c:.4f}  L_c={L_c:.4f}  r_a={r_a:.4f}  L_a={L_a:.4f}  [m]")
print("=" * 52)

# 3 · Solver C++: verificación y convergencia de malla

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

plt.rcParams.update({
    "text.usetex": False, "font.family": "serif", "font.serif": ["Times New Roman"],
    "font.size": 16, "axes.titlesize": 18, "axes.labelsize": 18,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 13,
    "axes.linewidth": 1.5, "grid.alpha": 0.3
})

# El modulo compilado queda en build/ o en build/Release/ segun el generador de CMake.
ruta_release = next(
    (p for p in (os.path.abspath("../preliminar-cpp/build/Release"),
                 os.path.abspath("../preliminar-cpp/build"))
     if os.path.isdir(p)),
    os.path.abspath("../preliminar-cpp/build"),
)
if ruta_release not in sys.path:
    sys.path.append(ruta_release)
import motor_mpd_cpp


a_cm = A_nozzle * 100.0   # cm  - divergencia de la tobera (= a_campana viejo)

#  EMPUJE SIMULADO (solver C++) para una malla dada
def empuje_simulado(Nr, Nz, I):
    malla = motor_mpd_cpp.Malla2D(Nr, Nz, rc, ra, Lc, La, True)
    R, Z  = malla.obtener_R(), malla.obtener_Z()
    campo = motor_mpd_cpp.CampoMagnetico(Nr, Nz)
    campo.calcular_campo_aplicado(B0, Lc, La, rc, ra, True)
    lorentz = motor_mpd_cpp.FuerzaLorentz(Nr, Nz)
    lorentz.calcular_tensores(I, Parametro_Hall, Lc, R, Z, campo.Br, campo.Bz)

    Fz_SI = np.nan_to_num(lorentz.Fz) * CM2_A_M2          # N/m^3
    dz_m  = (La / 100.0) / (Nz - 1)
    r_m   = R[:, :-1] / 100.0
    dr_m  = np.diff(R, axis=0)[:, :-1] / 100.0
    vol   = 2.0 * np.pi * r_m[:-1, :] * dr_m * dz_m       # m^3
    return np.sum(Fz_SI[:-1, :-1] * vol)                 # N

#  EMPUJE ANALÍTICO (modelo continuo, independiente de la malla)
L_c_m, a_m, rc_m, ra_m = Lc/100.0, a_cm/100.0, rc/100.0, ra/100.0

def integrando_Fz(z, I):
    dBz_dz = -3.0 * B0 * z / (a_m**2 * (1.0 + (z/a_m)**2)**2.5)
    rout   = ra_m * (1.0 + (z/a_m)**2)**0.75
    return -(Parametro_Hall * I / (4.0 * L_c_m)) * dBz_dz * (rout**2 - rc_m**2)

def empuje_analitico(I):
    val, _ = quad(integrando_Fz, 0.0, L_c_m, args=(I,))
    return val

#  PANEL A: empuje sim vs analítico en la malla de trabajo
Nr0, Nz0   = 30, 80
corrientes = np.linspace(200, 2000, 15)
T_ana = np.array([empuje_analitico(I) for I in corrientes])
T_sim = np.array([empuje_simulado(Nr0, Nz0, I) for I in corrientes])

#  PANEL B: convergencia de malla (error vs resolución, I fija)
I_conv = 1000.0
mallas = [(30,80),(45,120),(60,160),(90,240),(120,320),(180,480),(240,640)]
Nr_arr = np.array([m[0] for m in mallas])
T_ref  = empuje_analitico(I_conv)
errores = np.array([abs(empuje_simulado(nr, nz, I_conv) - T_ref) / abs(T_ref) * 100.0
                    for nr, nz in mallas])
ref_1er_orden = errores[0] * (Nr_arr[0] / Nr_arr)   # línea de referencia ~ 1/Nr

print("Convergencia de malla:")
for (nr, nz), err in zip(mallas, errores):
    print(f"  {nr:3d}x{nz:<4d}  error = {err:.3f} %")

#  FIGURA
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 9), constrained_layout=True)

# a) Empuje
ax1.plot(corrientes, T_ana, 'k--', lw=2, label='Modelo continuo (analítico)')
ax1.plot(corrientes, T_sim, 'o', ms=8, mfc='none', mec='blue', mew=2,
         label=f'Solver C++  ({Nr0}×{Nz0})')
ax1.set_xlabel(r"Corriente de arco $I$ [A]")
ax1.set_ylabel(r"Empuje axial $T$ [N]")
ax1.legend(loc='upper left', frameon=True, edgecolor='black')
ax1.grid(True, ls='--')
ax1.set_title("a) Empuje: solver vs. modelo continuo", fontweight='bold', loc='left')

# b) Convergencia (log-log)
ax2.loglog(Nr_arr, errores, 's-', color='red', ms=7, lw=1.5, label='Error de discretización')
ax2.loglog(Nr_arr, ref_1er_orden, ':', color='gray', lw=2, label=r'Primer orden ($\propto 1/N_r$)')
ax2.scatter([Nr0], [errores[0]], s=120, facecolors='none', edgecolors='darkred', lw=2, zorder=5)
ax2.annotate(f'{errores[0]:.2f}% @ {Nr0}×{Nz0}', xy=(Nr_arr[0], errores[0]),
             xytext=(Nr_arr[0]*1.15, errores[0]*1.35), color='darkred', fontweight='bold')
ax2.set_xlabel(r"Resolución radial $N_r$  ($N_z \propto N_r$)")
ax2.set_ylabel(r"Error relativo [%]")
ax2.legend(loc='upper right', frameon=True, edgecolor='black')
ax2.grid(True, which='both', ls='--')
ax2.set_title("b) Convergencia de malla", fontweight='bold', loc='left')

plt.savefig("validacion_empuje_paper.pdf", bbox_inches='tight')
plt.savefig("validacion_empuje_paper.png", dpi=600, bbox_inches='tight')
print("Figura exportada.")
plt.show()

# 4 · Topología de campo: tobera divergente vs. cilíndrica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FONT_SIZE_BASE = 16
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": FONT_SIZE_BASE,
    "axes.titlesize": FONT_SIZE_BASE + 2,
    "axes.labelsize": FONT_SIZE_BASE + 2,
    "xtick.labelsize": FONT_SIZE_BASE,
    "ytick.labelsize": FONT_SIZE_BASE,
})

# Geometría en cm: usa rc/ra/Lc/La de la celda de geometría.
# NO redefinir r_c/r_a/L_c/L_a (esos son los SI que usa la sim).
A_NOZZLE = A_nozzle * 100.0   # cm  (= 'a' del C++; A_nozzle del master está en m)
B0_tesla = B0                 # T   (del master)


nz, nr = 220, 220
z = np.linspace(0, La, nz)
r = np.linspace(0, 5.0, nr)
Zg, Rg = np.meshgrid(z, r)
denom  = 1.0 + (Zg / A_NOZZLE)**2
Bz     = B0_tesla / denom**1.5
dBz_dz = -3.0 * B0_tesla * Zg / (A_NOZZLE**2 * denom**2.5)
Br     = -0.5 * Rg * dBz_dz
Bmag   = np.sqrt(Bz**2 + Br**2)

# Ocultar el interior del cátodo
cath = (Rg < rc) & (Zg <= Lc)
Bz = np.where(cath, np.nan, Bz)
Br = np.where(cath, np.nan, Br)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 9))

def dibujar_panel(ax, pared_z, pared_r, titulo):
    # 1) Campo (al fondo)
    strm = ax.streamplot(z, r, Bz, Br, color=Bmag, cmap='plasma', density=1.4,
                         linewidth=1.2, norm=plt.Normalize(0, 0.5), arrowsize=0.8)
    # 2) Ánodo sólido cubriendo todo el dominio superior
    ax.fill_between(pared_z, pared_r, 5.0, color='darkslategray', zorder=5)
    ax.plot(pared_z, pared_r, color='black', lw=2.8, zorder=6)

    ax.fill_between([0, Lc], 0, rc, color='dimgray', zorder=10)
    ax.text(Lc/2, rc/2, 'Cátodo', color='white', ha='center', va='center',
            fontsize=FONT_SIZE_BASE - 5, zorder=11)
    ax.set_ylabel(r"Radio $r$ [cm]", labelpad=8)
    ax.set_xlim(0, La); ax.set_ylim(0, 4.9)
    ax.set_title(titulo, fontweight='bold', loc='left', fontsize=FONT_SIZE_BASE)
    return strm

zc = np.linspace(0, La, 120)
dibujar_panel(ax1, zc, np.full_like(zc, ra), "a) Configuración Cilíndrica")
pared_div = ra * (1.0 + (zc / A_NOZZLE)**2)**0.75   # = línea de campo
strm2 = dibujar_panel(ax2, zc, pared_div, "b) Configuración de Tobera Divergente")

ax1.set_xticklabels([])
ax2.set_xlabel(r"Posición axial $z$ [cm]", labelpad=10)

cbar = fig.colorbar(strm2.lines, ax=[ax1, ax2], orientation='vertical',
                    fraction=0.045, pad=0.02)
cbar.set_label(r"Magnitud del campo $|\mathbf{B}|$ [T]", labelpad=10)

plt.savefig("campos_simetricos.pdf", bbox_inches='tight')
plt.savefig("campos_simetricos.png", dpi=300, bbox_inches='tight')
print("Figura exportada.")
plt.show()

# 5 · Modelo de campos prescritos por regiones

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 100.0

# El modulo compilado queda en build/ o en build/Release/ segun el generador de CMake.
ruta_modulo = next(
    (p for p in (os.path.abspath("../preliminar-cpp/build/Release"),
                 os.path.abspath("../preliminar-cpp/build"))
     if os.path.isdir(p)),
    os.path.abspath("../preliminar-cpp/build"),
)
if ruta_modulo not in sys.path:
    sys.path.append(ruta_modulo)
import motor_mpd_cpp

# ============================================================
#  ILUSTRACIÓN DEL PROBLEMA DE CAMPOS (motor C++)
#  Cápsula AISLADA: malla + cm, con nombres propios para NO pisar
#  las variables del master (en SI). Lee solo rc/ra/Lc/La (cm) de
#  la celda de geometría; el resto es local.
# ============================================================
Nr, Nz   = 30, 80         # malla (exclusiva de la cápsula)
ramax    = 5.0            # cm  - radio máx. del ánodo  (local; NO el r_a_max del master)
Lsim     = 6.5            # cm  - largo del dominio      (local; NO el L_sim del master)
r_a_base = ra             # cm  - radio base del ánodo (= ra de geometría)
N_part   = 1000           # iones de la cápsula (no el N_iones del master)

# --- Knobs de los campos sintéticos (lo que ilustra el problema) ---
E_max           = 0.5     # campo radial inventado
E_max_z         = 10.0    # succión axial -Z
factor_escalado = 0.275   # escala la topología de Lorentz inyectada como E
Parametro_Hall  = 1.5     # knob de swirl
corriente_A     = I_lab   # A (del master)

# --- Malla de campos ---
R_1D = np.linspace(0, ramax, Nr)
Z_1D = np.linspace(0, Lsim, Nz)
R, Z = np.meshgrid(R_1D, Z_1D, indexing='ij')

Bz = np.zeros_like(R); Br = np.zeros_like(R)
Er = np.zeros_like(R); Ez = np.zeros_like(R)

# --- Campo radial sintético (aplasta iones hacia el centro) ---
mascara_region = (Z >= 0.1) & (Z <= Lc) & (R >= rc) & (R <= 0.8)
Er[mascara_region] = -E_max * (1.0 - (Z[mascara_region] / Lc))

# --- Campo axial sintético (-Z), aguas abajo del cátodo ---
mascara_punta = (Z >= Lc) & (Z <= La) & (R >= 0.0) & (R <= rc)
Ez[mascara_punta] = -E_max_z

# --- Campo magnético aplicado + tensores de Lorentz (C++) ---
campo = motor_mpd_cpp.CampoMagnetico(Nr, Nz)
campo.calcular_campo_aplicado(B0, Lc, La, rc, ramax, True)

lorentz = motor_mpd_cpp.FuerzaLorentz(Nr, Nz)
lorentz.calcular_tensores(corriente_A, Parametro_Hall, Lc, R, Z, campo.Br, campo.Bz)

Fz_base = np.nan_to_num(lorentz.Fz)
Fr_base = np.nan_to_num(lorentz.Fr)

# --- Inyección de la topología de Lorentz como campo E (empuje) ---
epsilon = 0.05
mascara_empuje = (Z >= 0.0) & (Z <= Lc) & (R >= (0.8 + epsilon)) & (R <= r_a_base)
Ez[mascara_empuje] = Fz_base[mascara_empuje] * factor_escalado
Er[mascara_empuje] = np.abs(Fr_base[mascara_empuje]) * factor_escalado

Ez = np.ascontiguousarray(Ez); Er = np.ascontiguousarray(Er)
Bz = np.ascontiguousarray(Bz); Br = np.ascontiguousarray(Br)

# --- Pared del ánodo ---
limite_pared = r_a_base + 0.15 * np.power(Z_1D, 2.0)
limite_pared = np.clip(limite_pared, 0, ramax)
limite_pared_c = np.ascontiguousarray(limite_pared)

# --- Inyección térmica (velocidad del sonido local, Argón neutro) ---
Te_eV = 1.5
T_gas_K = 300.0; gamma = 5.0 / 3.0
k_B = 1.38e-23; masa_ion_kg = 6.63e-26
u_z_inicial = np.sqrt((gamma * k_B * T_gas_K) / masa_ion_kg)

# --- Simulador C++ ---
solver = motor_mpd_cpp.SimuladorPIC(N_part, Te_eV, rc, ramax, Lc, La, Nr)
solver.inicializar_particulas(15000.0, r_a_base)

pos_ini = solver.obtener_posiciones_rz()
print(f"Posición inicial Ion 0: R={pos_ini[0,0]:.4f}, Z={pos_ini[0,1]:.4f}")
for _ in range(10):
    solver.avanzar_paso_temporal_boris(dt, Ez, Er, Bz, Br, limite_pared_c, Lsim, Nz)
pos_fin = solver.obtener_posiciones_rz()
print(f"Posición final Ion 0:   R={pos_fin[0,0]:.4f}, Z={pos_fin[0,1]:.4f}")
print("¡ÉXITO! Iones moviéndose." if abs(pos_fin[0,1]-pos_ini[0,1])>0 else "ERROR: iones estáticos.")

solver.inicializar_particulas(u_z_inicial, r_a_base)

# --- Animación ---
fig, ax1 = plt.subplots(figsize=(10, 5), facecolor='white')
ax1.set_xlim(0, Lsim); ax1.set_ylim(0, ramax)
ax1.set_xlabel('Posición Axial Z (cm)', fontsize=12)
ax1.set_ylabel('Radio R (cm)', fontsize=12)

ax1.fill_between([0, Lc], [0, 0], [rc, rc], color='dimgray', label='Cátodo')
ax1.fill_between(Z_1D, limite_pared, ramax, color='darkslategray', alpha=0.8, label='Ánodo')
ax1.axvline(x=Lc, color='red', linestyle='--', alpha=0.5, label='Punta (Z=L_c)')
ax1.fill_between([Lc, Lsim], [0, 0], [1.0, 1.0], color='yellow', alpha=0.15, label='Zona de Retorno')

scat_vivos  = ax1.scatter([], [], color='cyan',   s=10, zorder=5, label='En tránsito')
scat_barril = ax1.scatter([], [], color='red',    s=15, zorder=6, label='Colisión Barril')
scat_punta  = ax1.scatter([], [], color='purple', s=20, zorder=6, label='Colisión Punta')
scat_escape = ax1.scatter([], [], color='lime',   s=10, zorder=5, label='Eyectados')

ax1.legend(loc='upper right', bbox_to_anchor=(1.0, 1.15), ncol=5, fontsize=9)
titulo = ax1.set_title('Iniciando simulación...', fontsize=14, fontweight='bold', pad=25)

def update(frame):
    for _ in range(4000):
        solver.avanzar_paso_temporal_boris(dt, Ez, Er, Bz, Br, limite_pared_c, Lsim, Nz)
    pos = solver.obtener_posiciones_rz()
    estados = solver.obtener_estados()
    Zp = pos[:, 1] * 100.0; Rp = pos[:, 0] * 100.0
    m_vivos = estados == 0; m_barril = estados == 1
    m_punta = estados == 2; m_escape = estados == 3
    scat_vivos.set_offsets(np.column_stack((Zp[m_vivos],  Rp[m_vivos])))
    scat_barril.set_offsets(np.column_stack((Zp[m_barril], Rp[m_barril])))
    scat_punta.set_offsets(np.column_stack((Zp[m_punta],  Rp[m_punta])))
    scat_escape.set_offsets(np.column_stack((Zp[m_escape], Rp[m_escape])))
    tiempo_us = (frame * 500 * dt) * 1e6
    titulo.set_text(f"t = {tiempo_us:.2f} μs | Vivos: {np.sum(m_vivos)} | Escape: {np.sum(m_escape)} | "
                    f"Punta: {np.sum(m_punta)} | Barril: {np.sum(m_barril)}")
    return scat_vivos, scat_barril, scat_punta, scat_escape, titulo

print("Generando animación interactiva... ¡paciencia!")
anim = FuncAnimation(fig, update, frames=300, interval=40, blit=True)
plt.close(fig)
display(HTML(anim.to_jshtml()))

# 6 · Modelo test-particle de campos analíticos

In [ ]:
import numpy as np

def r_pared(z):
    return np.minimum(r_a*(1+(z/A_nozzle)**2)**0.75, r_a_max)

def simular(Ti_eV, V_vaina, N=4000, nsteps=400000, n_traj=70, cada_frame=CADA_FRAME, seed=1):
    np.random.seed(seed)
    rs=[]
    while len(rs)<N:
        rr=r_c+np.random.rand(N)*(r_a-r_c); rs.extend(rr[np.random.rand(N)<(r_c/rr)].tolist())
    r0=np.array(rs[:N]); th=np.random.rand(N)*2*np.pi
    x=r0*np.cos(th); y=r0*np.sin(th); z=np.random.rand(N)*L_a              # UNIFORME en el dominio
    vth=np.sqrt(e*Ti_eV/m_ion)
    u_iny=vth                                                             # haz axial = magnitud térmica
    vr=np.random.randn(N)*vth; vt=np.random.randn(N)*vth
    ex,ey=x/r0,y/r0
    vx=vr*ex - vt*ey; vy=vr*ey + vt*ex; vz=np.full(N, u_iny)              # anisótropa hacia adelante

    status=np.zeros(N,dtype=int)
    E_imp=np.zeros(N); zi=np.zeros(N); ri=np.zeros(N)
    traj=[[] for _ in range(n_traj)]; frames=[]; qm=e/m_ion

    for step in range(nsteps):
        if step%cada_frame==0:                                            # graba fotograma (animación)
            frames.append((z.copy()*100, np.sqrt(x*x+y*y)*100, status.copy()))
        if step%400==0:                                                   # graba trayectorias (figura)
            for j in range(n_traj):
                traj[j].append((z[j]*100, np.sqrt(x[j]**2+y[j]**2)*100))
        act=status==0
        if act.sum()<0.005*N: break
        idx=np.where(act)[0]
        xa,ya,za=x[idx],y[idx],z[idx]; vxa,vya,vza=vx[idx],vy[idx],vz[idx]
        r=np.maximum(np.sqrt(xa*xa+ya*ya),1e-9)
        Bz=B0/(1+(za/A_nozzle)**2)**1.5; dBz=-3*B0*za/A_nozzle**2/(1+(za/A_nozzle)**2)**2.5
        Br=-0.5*r*dBz; Bx=Br*xa/r; By=Br*ya/r
        Ex=np.zeros_like(xa);Ey=np.zeros_like(xa);Ez=np.zeros_like(xa)
        ib=(r>r_c)&(r<r_c+delta_vaina)&(za>0)&(za<L_c); Eb=V_vaina/delta_vaina
        Ex[ib]=-Eb*xa[ib]/r[ib]; Ey[ib]=-Eb*ya[ib]/r[ib]
        it=(za>L_c)&(za<L_c+delta_vaina)&(r<r_c); Ez[it]=-V_vaina/delta_vaina
        vmx=vxa+qm*Ex*dt/2;vmy=vya+qm*Ey*dt/2;vmz=vza+qm*Ez*dt/2
        tx=qm*Bx*dt/2;ty=qm*By*dt/2;tz=qm*Bz*dt/2;t2=tx*tx+ty*ty+tz*tz
        sx=2*tx/(1+t2);sy=2*ty/(1+t2);sz=2*tz/(1+t2)
        vpx=vmx+(vmy*tz-vmz*ty);vpy=vmy+(vmz*tx-vmx*tz);vpz=vmz+(vmx*ty-vmy*tx)
        vxn=vmx+(vpy*sz-vpz*sy);vyn=vmy+(vpz*sx-vpx*sz);vzn=vmz+(vpx*sy-vpy*sx)
        vxn+=qm*Ex*dt/2;vyn+=qm*Ey*dt/2;vzn+=qm*Ez*dt/2
        xn=xa+vxn*dt;yn=ya+vyn*dt;zn=za+vzn*dt;rn=np.sqrt(xn*xn+yn*yn)
        Ek=0.5*m_ion*(vxn**2+vyn**2+vzn**2)/e
        st=np.zeros(len(idx),dtype=int)
        st[zn>=L_sim]=3
        st[(zn<=0)&(st==0)]=5
        st[(rn>=r_pared(zn))&(zn>=0)&(zn<=L_a)&(st==0)]=4
        cat=(rn<=r_c)&(zn>0)&(zn<=L_c)&(st==0); side=r>r_c
        st[cat&side]=1; st[cat&(~side)]=2
        st[(za>L_c)&(zn<=L_c)&(rn<r_c)&(st==0)]=2
        x[idx]=xn;y[idx]=yn;z[idx]=zn;vx[idx]=vxn;vy[idx]=vyn;vz[idx]=vzn
        hit=(st==1)|(st==2)
        status[idx[st>0]]=st[st>0]
        E_imp[idx[hit]]=Ek[hit]; zi[idx[hit]]=zn[hit]; ri[idx[hit]]=np.minimum(rn[hit],r_c)

    frames.append((z.copy()*100, np.sqrt(x*x+y*y)*100, status.copy()))    # fotograma final
    me=(status==1)|(status==2)
    return dict(status=status, E_imp=E_imp, zi=zi, ri=ri, traj=traj, frames=frames, N=N,
        barril=100*(status==1).sum()/N, punta=100*(status==2).sum()/N,
        escape=100*(status==3).sum()/N, anodo=100*(status==4).sum()/N,
        backplate=100*(status==5).sum()/N, activo=100*(status==0).sum()/N,
        E_med=E_imp[me].mean() if me.any() else 0, carga=E_imp[me].sum())

R = simular(T_i, V_vaina)
R = simular(T_i, V_vaina)
print(f"barril {R['barril']:.1f}%   punta {R['punta']:.1f}%   escape {R['escape']:.1f}%   "
      f"anodo {R['anodo']:.1f}%   backplate {R['backplate']:.1f}%   activo {R['activo']:.1f}%")
print(f"Energia de impacto media: {R['E_med']:.1f} eV    carga termica relativa: {R['carga']:.0f}")
print(f"{len(R['frames'])} fotogramas grabados (una sola corrida)")

# 7 · Diagnóstico: trayectorias, bombardeo y energía de impacto

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig=plt.figure(figsize=(15.5,5.8))
gs=fig.add_gridspec(1,3,width_ratios=[2,2,1.2],wspace=0.38)
ax1=fig.add_subplot(gs[0]); ax2=fig.add_subplot(gs[1]); ax3=fig.add_subplot(gs[2])
zz=np.linspace(0,L_sim,300)

# Panel 1: trayectorias + geometria + lineas de campo
for r0f in [0.4,0.8,1.2,1.6,2.0,2.65]:
    ax1.plot(zz*100,r0f*(1+(zz/A_nozzle)**2)**0.75,color='0.86',lw=0.8,zorder=1)
ax1.fill_between(zz*100,r_pared(zz)*100,r_a_max*100+0.5,color='#5a6b7a',zorder=2)
ax1.text(5.2,4.3,'ANODO',color='white',fontsize=9,weight='bold',zorder=3)
ax1.add_patch(plt.Rectangle((0,0),L_c*100,r_c*100,color='0.2',zorder=6))
ax1.text(1.4,0.13,'catodo',color='white',fontsize=7,zorder=7,ha='center')
col={1:'#d62728',2:'#7a0010',3:'#6aa8de',4:'#9467bd',5:'#888888',0:'#cccccc'}
lab={1:'choca barril',2:'choca punta',3:'escapa',4:'choca anodo',5:'backplate'}
seen=set()
for j,t in enumerate(R['traj']):
    if len(t)<2: continue
    arr=np.array(t); fate=R['status'][j]
    lb=lab.get(fate) if fate not in seen else None
    if fate in lab: seen.add(fate)
    ax1.plot(arr[:,0],arr[:,1],color=col[fate],lw=0.6,alpha=0.6,zorder=4,label=lb)
ax1.set_xlim(0,L_sim*100);ax1.set_ylim(0,r_a_max*100)
ax1.set_xlabel('Posicion axial z (cm)');ax1.set_ylabel('Radio r (cm)')
ax1.set_title('Trayectorias de Ar$^+$ (test-particle, sin colisiones)')
ax1.legend(loc='upper left',fontsize=8,framealpha=0.92)

# Panel 2: bombardeo del catodo, color = energia
ax2.add_patch(plt.Rectangle((0,0),L_c*100,r_c*100,color='0.2',zorder=6))
me=(R['status']==1)|(R['status']==2)
sc=ax2.scatter(R['zi'][me]*100,R['ri'][me]*100,c=R['E_imp'][me],cmap='inferno',
               s=26,zorder=8,vmin=0,vmax=35,edgecolors='none')
ax2.axvline(L_c*100,color='r',ls='--',lw=1,zorder=7)
ax2.text(L_c*100+0.1,0.46,'punta',color='r',fontsize=8,rotation=90,va='top')
ax2.set_xlim(0,L_sim*100);ax2.set_ylim(0,0.55)
ax2.set_xlabel('Posicion axial z (cm)');ax2.set_ylabel('Radio r (cm)')
ax2.set_title(f'Bombardeo del catodo ({me.sum()} iones, {100*me.sum()/R["N"]:.0f}%)')
cb=plt.colorbar(sc,ax=ax2,fraction=0.046,pad=0.03); cb.set_label('Energia (eV)')

# Panel 3: histograma de energia
ax3.hist(R['E_imp'][me],bins=28,color='#d62728',alpha=0.82,orientation='horizontal')
ax3.axhline(30,color='k',ls=':',lw=1.3,label='umbral sputter\nAr$\\to$W ~30 eV')
ax3.set_ylabel('Energia de impacto (eV)');ax3.set_xlabel('N de iones')
ax3.set_title('Energia de impacto');ax3.legend(fontsize=8,loc='upper right');ax3.set_ylim(0,38)

plt.suptitle(f'Simulacion base (sin HTS)   |   B$_0$={B0} T,  T$_i$={T_i} eV,  caida de vaina={V_vaina:.0f} V   '
             f'|   escape {R["escape"]:.0f}%,  catodo {R["barril"]+R["punta"]:.0f}%,  anodo {R["anodo"]:.1f}%',
             fontsize=10.5,y=1.01)
plt.tight_layout(); plt.show()

# 8 · Animación del modelo test-particle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

Z_1D = np.linspace(0, L_sim, 300); limite_pared = r_pared(Z_1D)
fig, ax1 = plt.subplots(figsize=(10, 5), facecolor='white')
ax1.set_xlim(0, L_sim*100); ax1.set_ylim(0, r_a_max*100)
ax1.set_xlabel('Posición Axial Z (cm)', fontsize=12); ax1.set_ylabel('Radio R (cm)', fontsize=12)
ax1.fill_between([0, L_c*100], [0, 0], [r_c*100, r_c*100], color='dimgray', label='Cátodo')
ax1.fill_between(Z_1D*100, limite_pared*100, r_a_max*100, color='darkslategray', alpha=0.8, label='Ánodo')
ax1.axvline(x=L_c*100, color='red', linestyle='--', alpha=0.5, label='Punta (Z=L_c)')
scat_vivos  = ax1.scatter([], [], color='cyan',   s=10, zorder=5, label='En tránsito')
scat_barril = ax1.scatter([], [], color='red',    s=15, zorder=6, label='Colisión Barril')
scat_punta  = ax1.scatter([], [], color='purple', s=20, zorder=6, label='Colisión Punta')
scat_escape = ax1.scatter([], [], color='lime',   s=10, zorder=5, label='Eyectados')
scat_anodo  = ax1.scatter([], [], color='orange', s=12, zorder=6, label='Colisión Ánodo')
ax1.legend(loc='upper right', bbox_to_anchor=(1.0, 1.15), ncol=6, fontsize=9)
titulo = ax1.set_title('', fontsize=14, fontweight='bold', pad=25)

def update(frame):
    Zp, Rp, est = R['frames'][frame]                      # solo reproduce los fotogramas de la Celda 1
    scat_vivos.set_offsets(np.column_stack((Zp[est==0], Rp[est==0])))
    scat_barril.set_offsets(np.column_stack((Zp[est==1], Rp[est==1])))
    scat_punta.set_offsets(np.column_stack((Zp[est==2], Rp[est==2])))
    scat_escape.set_offsets(np.column_stack((Zp[est==3], Rp[est==3])))
    scat_anodo.set_offsets(np.column_stack((Zp[est==4], Rp[est==4])))
    tiempo_us = (frame * CADA_FRAME * dt) * 1e6
    titulo.set_text(f"t = {tiempo_us:.2f} μs | Vivos: {np.sum(est==0)} | Escape: {np.sum(est==3)} | "
                    f"Punta: {np.sum(est==2)} | Barril: {np.sum(est==1)}")
    return scat_vivos, scat_barril, scat_punta, scat_escape, scat_anodo, titulo

print("Generando animación...")
anim = FuncAnimation(fig, update, frames=range(1, len(R['frames'])), interval=40, blit=True)  # <- arranca en 1
plt.close(fig)
display(HTML(anim.to_jshtml()))

# 9 · HTS: refuerzo de campo de tobera uniforme

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

N_SWEEP = 2000   # iones por corrida (sube a 4000 para figuras finales; baja a 1000 si va lento)

def _coil(R, Z, Bp, z0, a):
    """Campo en eje + Br paraxial de una espira de radio a centrada en z0 (SI)."""
    dz = Z - z0
    Bz = Bp/(1+(dz/a)**2)**1.5
    dBz = -3*Bp*dz/(a**2*(1+(dz/a)**2)**2.5)
    return Bz, -0.5*R*dBz

def simular_hts(B0, cusp_B=0.0, cusp_delta=1.0e-2, cusp_a=1.5e-2, N=N_SWEEP, nsteps=300000, seed=1):
    """Test-particle: tobera (espira en z=0, intensidad B0) + cusp opcional
       (dos espiras opuestas en z=L_c ± cusp_delta). Devuelve fracciones por destino."""
    np.random.seed(seed)
    rs=[]
    while len(rs)<N:
        rr=r_c+np.random.rand(N)*(r_a-r_c); rs.extend(rr[np.random.rand(N)<(r_c/rr)].tolist())
    r0=np.array(rs[:N]); th=np.random.rand(N)*2*np.pi
    x=r0*np.cos(th); y=r0*np.sin(th); z=np.random.rand(N)*L_a
    vth=np.sqrt(e*T_i/m_ion); u_iny=vth
    vr=np.random.randn(N)*vth; vt=np.random.randn(N)*vth
    ex,ey=x/r0,y/r0
    vx=vr*ex-vt*ey; vy=vr*ey+vt*ex; vz=np.full(N,u_iny)
    status=np.zeros(N,dtype=int); E_imp=np.zeros(N); qm=e/m_ion
    for step in range(nsteps):
        act=status==0
        if act.sum()<0.005*N: break
        idx=np.where(act)[0]
        xa,ya,za=x[idx],y[idx],z[idx]; vxa,vya,vza=vx[idx],vy[idx],vz[idx]
        r=np.maximum(np.sqrt(xa*xa+ya*ya),1e-9)
        Bz=B0/(1+(za/A_nozzle)**2)**1.5; dBz=-3*B0*za/A_nozzle**2/(1+(za/A_nozzle)**2)**2.5
        Br=-0.5*r*dBz
        if cusp_B!=0.0:
            Bz1,Br1=_coil(r,za,+cusp_B,L_c-cusp_delta,cusp_a)
            Bz2,Br2=_coil(r,za,-cusp_B,L_c+cusp_delta,cusp_a)
            Bz=Bz+Bz1+Bz2; Br=Br+Br1+Br2
        Bx=Br*xa/r; By=Br*ya/r
        Ex=np.zeros_like(xa);Ey=np.zeros_like(xa);Ez=np.zeros_like(xa)
        ib=(r>r_c)&(r<r_c+delta_vaina)&(za>0)&(za<L_c); Eb=V_vaina/delta_vaina
        Ex[ib]=-Eb*xa[ib]/r[ib]; Ey[ib]=-Eb*ya[ib]/r[ib]
        it=(za>L_c)&(za<L_c+delta_vaina)&(r<r_c); Ez[it]=-V_vaina/delta_vaina
        vmx=vxa+qm*Ex*dt/2;vmy=vya+qm*Ey*dt/2;vmz=vza+qm*Ez*dt/2
        tx=qm*Bx*dt/2;ty=qm*By*dt/2;tz=qm*Bz*dt/2;t2=tx*tx+ty*ty+tz*tz
        sx=2*tx/(1+t2);sy=2*ty/(1+t2);sz=2*tz/(1+t2)
        vpx=vmx+(vmy*tz-vmz*ty);vpy=vmy+(vmz*tx-vmx*tz);vpz=vmz+(vmx*ty-vmy*tx)
        vxn=vmx+(vpy*sz-vpz*sy);vyn=vmy+(vpz*sx-vpx*sz);vzn=vmz+(vpx*sy-vpy*sx)
        vxn+=qm*Ex*dt/2;vyn+=qm*Ey*dt/2;vzn+=qm*Ez*dt/2
        xn=xa+vxn*dt;yn=ya+vyn*dt;zn=za+vzn*dt;rn=np.sqrt(xn*xn+yn*yn)
        Ek=0.5*m_ion*(vxn**2+vyn**2+vzn**2)/e
        st=np.zeros(len(idx),dtype=int)
        st[zn>=L_sim]=3; st[(zn<=0)&(st==0)]=5
        st[(rn>=r_pared(zn))&(zn>=0)&(zn<=L_a)&(st==0)]=4
        cat=(rn<=r_c)&(zn>0)&(zn<=L_c)&(st==0); side=r>r_c
        st[cat&side]=1; st[cat&(~side)]=2
        st[(za>L_c)&(zn<=L_c)&(rn<r_c)&(st==0)]=2
        x[idx]=xn;y[idx]=yn;z[idx]=zn;vx[idx]=vxn;vy[idx]=vyn;vz[idx]=vzn
        hit=(st==1)|(st==2); status[idx[st>0]]=st[st>0]; E_imp[idx[hit]]=Ek[hit]
    f=lambda k:100*(status==k).sum()/N
    me=(status==1)|(status==2)
    return dict(escape=f(3), barril=f(1), punta=f(2), anodo=f(4),
                backplate=f(5), atrapados=f(0),
                E_med=E_imp[me].mean() if me.any() else 0.0)

# ===== Barrido de intensidad de campo: cobre (~0.5 T) -> HTS (~5 T) =====
valores_B0 = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
print("Barrido de B0..."); res_B0 = [simular_hts(b, cusp_B=0.0) for b in tqdm(valores_B0)]

esc=[r['escape'] for r in res_B0]; bar=[r['barril'] for r in res_B0]
pun=[r['punta'] for r in res_B0]; ano=[r['anodo'] for r in res_B0]

fig, ax = plt.subplots(figsize=(9,5), facecolor='white')
ax.plot(valores_B0, esc, 'o-', color='lime',   label='Escape (propulsión útil)')
ax.plot(valores_B0, bar, 's-', color='red',    label='Colisión barril (erosión)')
ax.plot(valores_B0, pun, '^-', color='purple', label='Colisión punta (erosión)')
ax.plot(valores_B0, ano, 'D-', color='gray',   label='Colisión ánodo')
ax.set_xlabel('Campo de tobera B₀ (T)', fontsize=12)
ax.set_ylabel('Fracción de iones (%)', fontsize=12)
ax.set_title('Erosión y propulsión vs. intensidad de campo\n'
             'Cobre (~0.5 T) → HTS (~5 T)  |  Ar⁺, Tᵢ = 1.5 eV', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n{'B0 (T)':<9}{'Escape':<9}{'Barril':<9}{'Punta':<8}{'Ánodo':<8}{'Backpl':<8}")
print("-"*49)
for b,r in zip(valores_B0,res_B0):
    print(f"{b:<9.2f}{r['escape']:<9.1f}{r['barril']:<9.1f}{r['punta']:<8.1f}{r['anodo']:<8.1f}{r['backplate']:<8.1f}")

# 10 · HTS: topología de cusp magnético

In [ ]:
# ===== Fuerza del cusp (tobera fija 0.5 T + dos espiras opuestas en z = L_c ± 1 cm) =====
valores_cusp = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
print("Barrido de cusp..."); res_cusp = [simular_hts(0.5, cusp_B=cb) for cb in tqdm(valores_cusp)]

esc=[r['escape'] for r in res_cusp]; bar=[r['barril'] for r in res_cusp]
pun=[r['punta'] for r in res_cusp]; ano=[r['anodo'] for r in res_cusp]; bpl=[r['backplate'] for r in res_cusp]

fig, ax = plt.subplots(figsize=(9,5), facecolor='white')
ax.plot(valores_cusp, esc, 'o-', color='lime',   label='Escape (propulsión útil)')
ax.plot(valores_cusp, bar, 's-', color='red',    label='Colisión barril (erosión)')
ax.plot(valores_cusp, pun, '^-', color='purple', label='Colisión punta (erosión)')
ax.plot(valores_cusp, ano, 'D-', color='gray',   label='Colisión ánodo')
ax.plot(valores_cusp, bpl, 'v-', color='orange', label='Reflejados atrás (espejo del cusp)')
ax.set_xlabel('Campo del cusp en la punta  B_HTS (T)', fontsize=12)
ax.set_ylabel('Fracción de iones (%)', fontsize=12)
ax.set_title('Topología de cusp en la punta del cátodo (tobera 0.5 T fija)\n'
             'dos espiras opuestas en z = L_c ± 1 cm', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n{'B_HTS (T)':<11}{'Escape':<9}{'Barril':<9}{'Punta':<8}{'Ánodo':<8}{'Backpl':<8}{'Atrap':<8}")
print("-"*59)
for b,r in zip(valores_cusp,res_cusp):
    print(f"{b:<11.1f}{r['escape']:<9.1f}{r['barril']:<9.1f}{r['punta']:<8.1f}{r['anodo']:<8.1f}{r['backplate']:<8.1f}{r['atrapados']:<8.1f}")

# 11 · Comparación de las tres topologías

In [ ]:
# ===== Sin HTS  vs  HTS uniforme  vs  HTS cusp =====
configs = {
    'Sin HTS\n(cobre, 0.5 T)': res_B0[valores_B0.index(0.5)],
    'HTS uniforme\n(5 T)':      res_B0[valores_B0.index(5.0)],
    'HTS cusp\n(0.5 T + 1 T)':  res_cusp[valores_cusp.index(1.0)],
}
col_cfg = ['#808080', '#2ca02c', '#d62728']   # gris / verde / rojo

# Carga térmica al cátodo = (% que golpea el cátodo) x (energía media de impacto).
# Proxy del flujo de energía depositada; la erosión real es por evaporación (ver texto).
carga     = {lab: (r['barril']+r['punta'])*r['E_med'] for lab,r in configs.items()}
ref       = list(carga.values())[0]                    # sin HTS = 100%
carga_rel = {lab: 100*c/ref for lab,c in carga.items()}

metricas = ['escape','barril','punta','anodo','backplate']
nombres  = ['Escape','Barril','Punta','Ánodo','Reflejados']

fig = plt.figure(figsize=(13, 5.5), facecolor='white')
gs  = fig.add_gridspec(1, 2, width_ratios=[3, 1], wspace=0.28)
ax1 = fig.add_subplot(gs[0]); ax2 = fig.add_subplot(gs[1])

x = np.arange(len(metricas)); w = 0.26
for i,(lab,r) in enumerate(configs.items()):
    ax1.bar(x+(i-1)*w, [r[mt] for mt in metricas], w, label=lab, color=col_cfg[i])
ax1.set_xticks(x); ax1.set_xticklabels(nombres, fontsize=11)
ax1.set_ylabel('Fracción de iones (%)', fontsize=12)
ax1.set_title('Destino de los iones', fontsize=12)
ax1.legend(fontsize=9); ax1.grid(True, axis='y', alpha=0.3)

ax2.bar(range(3), list(carga_rel.values()), color=col_cfg, width=0.6)
ax2.axhline(100, color='k', ls=':', lw=1, alpha=0.5)
ax2.set_xticks(range(3)); ax2.set_xticklabels(['Sin\nHTS','HTS\nunif.','HTS\ncusp'], fontsize=9)
ax2.set_ylabel('Carga térmica al cátodo\n(relativa, sin HTS = 100%)', fontsize=11)
ax2.set_title('Carga térmica', fontsize=12)
for i,v in enumerate(carga_rel.values()):
    ax2.text(i, v+3, f'{v:.0f}%', ha='center', fontsize=10, fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)

fig.suptitle('Comparación de topologías de campo  (modelo test-particle, Ar⁺)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

print(f"\n{'Configuración':<22}{'Escape':<9}{'Barril':<9}{'Punta':<8}{'Ánodo':<8}{'Backpl':<8}{'E_imp(eV)':<11}{'Carga rel.'}")
print("-"*92)
for lab,r in configs.items():
    l = lab.replace('\n',' ')
    print(f"{l:<22}{r['escape']:<9.1f}{r['barril']:<9.1f}{r['punta']:<8.1f}"
          f"{r['anodo']:<8.1f}{r['backplate']:<8.1f}{r['E_med']:<11.1f}{carga_rel[l[:0]+lab]:.0f}%" if False else
          f"{l:<22}{r['escape']:<9.1f}{r['barril']:<9.1f}{r['punta']:<8.1f}"
          f"{r['anodo']:<8.1f}{r['backplate']:<8.1f}{r['E_med']:<11.1f}{carga_rel[lab]:.0f}%")